# MixMatch: one loss that mixes labeled and guessed-label data

_Semi & Self-Supervised Learning_

**Guess a label for each unlabeled image, sharpen the guess, then MixUp everything into one clean supervised + consistency loss.**

You have a few labeled images and a big pile of unlabeled ones. MixMatch (Berthelot et al., 2019) is a recipe that turns that pile into useful training signal with one combined loss.

       The trick is to guess a soft label for each unlabeled image, make that guess a little more confident (sharpen it), and then blend labeled and guessed-label images together with MixUp so the network never sees a hard boundary between "real label" and "guess".

---

This notebook is a practice scaffold for the **MixMatch: one loss that mixes labeled and guessed-label data** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def sharpen(p, T):
    # p: (B, C) averaged guessed distribution ; raise to 1/T and renormalize
    pt = p ** (1.0 / T)
    return pt / pt.sum(dim=1, keepdim=True)

def mixup(x1, p1, x2, p2, alpha):
    # convex blend of inputs and of their (soft) labels; keep first sample dominant
    lam = torch.distributions.Beta(alpha, alpha).sample().to(x1.device)
    lam = torch.max(lam, 1.0 - lam)
    x = lam * x1 + (1.0 - lam) * x2
    p = lam * p1 + (1.0 - lam) * p2
    return x, p

def mixmatch_step(model, augment, x_l, y_l, x_u,
                  K=2, T=0.5, alpha=0.75, lambda_u=75.0, num_classes=10):
    # x_l: labeled images ; y_l: int labels ; x_u: unlabeled images
    model.train()
    y_l_onehot = F.one_hot(y_l, num_classes).float()

    # --- step 1+2: guess a label by averaging K augmentations, then sharpen ---
    with torch.no_grad():
        u_aug = [augment(x_u) for _ in range(K)]           # K random augmentations
        logits = [model(u) for u in u_aug]
        avg = sum(F.softmax(l, dim=1) for l in logits) / K  # average distribution
        q = sharpen(avg, T)                                 # sharpened guessed label
        q = q.detach()

    # build the two streams: labeled (real one-hot) + unlabeled views (guessed q)
    xb_l, pb_l = x_l, y_l_onehot
    xb_u = torch.cat(u_aug, dim=0)
    pb_u = torch.cat([q for _ in range(K)], dim=0)

    # --- step 3: MixUp everything against a shuffled copy of the pooled batch ---
    all_x = torch.cat([xb_l, xb_u], dim=0)
    all_p = torch.cat([pb_l, pb_u], dim=0)
    perm = torch.randperm(all_x.size(0), device=all_x.device)
    mx, mp = mixup(all_x, all_p, all_x[perm], all_p[perm], alpha)

    n_l = xb_l.size(0)
    mx_l, mp_l = mx[:n_l], mp[:n_l]      # mixed examples whose first source was labeled
    mx_u, mp_u = mx[n_l:], mp[n_l:]      # mixed examples whose first source was unlabeled

    # --- step 4: supervised cross-entropy + unlabeled MSE consistency ---
    logits_l = model(mx_l)
    loss_x = -(mp_l * F.log_softmax(logits_l, dim=1)).sum(dim=1).mean()  # soft cross-entropy

    probs_u = F.softmax(model(mx_u), dim=1)
    loss_u = F.mse_loss(probs_u, mp_u)                                   # consistency (MSE)

    return loss_x + lambda_u * loss_u    # backward() + optimizer.step() on this

## Visualize the data & results

_What does sharpening do to a real, ambiguous predicted distribution — and does leaning on unlabeled data beat labels-only?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading

d = load_digits()                       # 1797 real 8x8 handwritten digits
X = d.data / 16.0                        # pixel intensities scaled to 0..1
y = d.target

# ---- panel 1: sharpening a real ambiguous softmax ----
clf = LogisticRegression(max_iter=2000, C=1.0).fit(X[:1500], y[:1500])
probs = clf.predict_proba(X[1500:])
ent = -(probs * np.log(probs + 1e-12)).sum(1)   # entropy of each held-out prediction
i = 1500 + int(np.argsort(ent)[-15])            # pick a high-entropy (ambiguous) digit
p = clf.predict_proba(X[i:i+1])[0]              # its predicted distribution
def sharpen(p, T):
    q = p ** (1.0 / T)
    return q / q.sum()
print("true label", y[i])
for T in [1.0, 0.5, 0.25]:
    s = sharpen(p, T)
    print("T=%.2f  class1=%.3f  class6=%.3f" % (T, s[1], s[6]))

# ---- panel 2: semi-supervised vs labels-only across label budgets ----
rng = np.random.RandomState(1)
perm = rng.permutation(len(y))
Xtr, ytr = X[perm[:1400]], y[perm[:1400]]
Xte, yte = X[perm[1400:]], y[perm[1400:]]
for nl in [10, 20, 40, 80, 160]:
    sup = LogisticRegression(max_iter=2000).fit(Xtr[:nl], ytr[:nl]).score(Xte, yte)
    yl = np.copy(ytr); yl[nl:] = -1             # mask all but first nl as unlabeled
    ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2).fit(Xtr, yl)
    print("labels=%d  labels-only=%.3f  +unlabeled=%.3f" % (nl, sup, ls.score(Xte, yte)))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Sharpen the guess $p=[0.6,\,0.4]$ with temperature $T=0.5$. What do you get?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the power: $1/T = 1/0.5 = 2$. — _Sharpening raises each probability to the power $1/T$._
- Raise each entry to the power 2: $0.6^2 = 0.36$, $0.4^2 = 0.16$. — _Bigger probabilities grow faster than small ones, which makes the distribution peakier._
- Renormalize: divide each by the sum $0.36 + 0.16 = 0.52$. — _A probability distribution must still sum to 1 after sharpening._

**Answer:** $0.36/0.52 \approx 0.692$ and $0.16/0.52 \approx 0.308$, so $\text{Sharpen}(p,0.5) \approx [0.692,\,0.308]$. The top class rose from 0.60 to 0.69 — more confident, but not collapsed to a hard 1.0, which is the point of using $T=0.5$ rather than $T\to 0$.

</details>

**Problem 2.** Why does MixMatch average $K$ augmented predictions before sharpening, instead of sharpening a single prediction?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall each augmentation produces a slightly different, noisy distribution. — _A random crop or flip perturbs the prediction a little each time._
- Recall that averaging several noisy estimates reduces their variance. — _The random fluctuations partly cancel, leaving a steadier estimate._
- Note that sharpening commits confidently to whatever it is given. — _If you sharpen a noisy single guess, you commit hard to that noise._

**Answer:** Averaging the $K$ distributions cancels much of the per-augmentation noise, so $\bar{p}$ is a more reliable guess than any single view. Sharpening then commits confidently to a cleaner target, which is more likely correct. Sharpening one noisy prediction would instead lock the model onto whatever random error that single augmentation produced.

</details>

**Problem 3.** A teammate sets the unlabeled-loss weight $\lambda_U$ to its full value from step 1 of training. What goes wrong, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the guessed labels look like at step 1. — _At the start the network is barely trained, so its predicted distributions are nearly random._
- Trace what a large $\lambda_U$ does to those guesses. — _$\lambda_U$ scales how strongly the consistency loss pulls predictions toward the guessed labels._

**Answer:** At step 1 the guesses are essentially random, so a full-strength $\mathcal{L}_U$ trains the model hard to match noise — it learns garbage and may never recover. The fix is to ramp $\lambda_U$ up from 0 over many training steps, so the model first learns from the real labeled data and only gradually starts trusting its own (now better) guesses.

</details>